# 📊 GolStats — Power BI Preparation

## Objective

The objective of this notebook is to prepare and validate the Gold layer datasets before connecting them to Power BI.

This phase focuses on:

1. Reviewing the final Gold tables.
2. Inspecting schemas and data granularity.
3. Identifying fact and dimension tables.
4. Preparing the analytical data model.
5. Validating keys and relationships.
6. Creating any additional datasets required for Power BI.

The final goal is to provide a clean and reliable analytical model for the GolStats Power BI dashboard.

---

## Source Tables

The Power BI model will initially use the following Gold layer tables:

- `golstats.gold.partidos`
- `golstats.gold.estadisticas_equipo`
- `golstats.gold.estadisticas_jugador`
- `golstats.gold.player_tournament`

All source tables were validated in the previous pipeline stage:

```text
GOLD LAYER PRODUCTION-READY: True

In [0]:

from pyspark.sql import functions as F

gold_tables = [
    "golstats.gold.partidos",
    "golstats.gold.estadisticas_equipo",
    "golstats.gold.estadisticas_jugador",
    "golstats.gold.player_tournament"
]

for table_name in gold_tables:
    
    print("\n" + "=" * 80)
    print(f"TABLE: {table_name}")
    print("=" * 80)
    
    df = spark.table(table_name)
    
    print("\n📐 SCHEMA")
    df.printSchema()
    
    print("\n📊 ROW COUNT")
    print(df.count())
    
    print("\n👀 SAMPLE DATA")
    display(df.limit(5))

![image_1788650259754.png](./image_1788650259754.png "image_1788650259754.png")

In [0]:
# Dim_Equipo — un registro por equipo
dim_equipo = (
    spark.table("golstats.gold.estadisticas_equipo")
    .select("team")
    .distinct()
    .withColumnRenamed("team", "team_name")
)
dim_equipo.write.format("delta").mode("overwrite").saveAsTable("golstats.gold.dim_equipo")

# Dim_Jugador — un registro por jugador
dim_jugador = (
    spark.table("golstats.gold.player_tournament")
    .select("player_id", "player")
    .distinct()
)
dim_jugador.write.format("delta").mode("overwrite").saveAsTable("golstats.gold.dim_jugador")

In [0]:
from pyspark.sql.functions import col

df_partidos = spark.table("golstats.gold.partidos")

huerfanos_home = df_partidos.join(dim_equipo, col("home_team") == col("team_name"), "left_anti").count()
huerfanos_away = df_partidos.join(dim_equipo, col("away_team") == col("team_name"), "left_anti").count()
huerfanos_jugador = (
    spark.table("golstats.gold.estadisticas_jugador")
    .join(dim_jugador, "player_id", "left_anti")
    .count()
)

print("Home teams sin match en dim_equipo:", huerfanos_home)   # debería ser 0
print("Away teams sin match en dim_equipo:", huerfanos_away)   # debería ser 0
print("Jugadores sin match en dim_jugador:", huerfanos_jugador) # debería ser 0

## Data Model — Final Design

**Fact tables:**
- `golstats.gold.partidos` (1 row per match)
- `golstats.gold.estadisticas_equipo` (1 row per team-match)
- `golstats.gold.estadisticas_jugador` (1 row per player-match)
- `golstats.gold.player_tournament` (1 row per player, tournament-level aggregate — standalone, not related to the match-level facts)

**Dimension tables:**
- `golstats.gold.dim_equipo` (team_name)
- `golstats.gold.dim_jugador` (player_id, player)

**Relationships:**
- `dim_equipo.team_name` → `estadisticas_equipo.team`
- `estadisticas_equipo.match_id` → `partidos.match_id`
- `dim_jugador.player_id` → `estadisticas_jugador.player_id`

**Referential integrity validated:** 0 orphaned home teams, 0 orphaned away teams, 0 orphaned players.

Data model is ready for Power BI.